<a href="https://colab.research.google.com/github/Mohana009/24P31A42D0_DriveReady_OPC_2026/blob/main/Day-2/task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai

In [3]:
# once per Colab session

from google.colab import userdata                # reads Colab Secrets
from openai import OpenAI                        # the client class

client = OpenAI(
    api_key=userdata.get("GROQ_API_KEY"),        # who you are, who pays
    base_url="https://api.groq.com/openai/v1",   # send requests to Groq, not OpenAI
)

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",                  # which model answers
    messages=[
        {"role": "system", "content": "You are a funny joker"},
        {"role": "user",   "content": "tell me a joke"},
    ],
)
print(response.choices[0].message.content)

Why don’t scientists trust atoms anymore?

Because they **make up** everything! 😄


In [4]:
SYSTEM = """ROLE:   You are the assistant for Sweet Crumbs Home Bakery.
FACTS:  We bake to order. Orders need 24 hours' notice. All cakes have eggless versions.
        Plum cake Rs 500. Chocolate cake Rs 650. Cupcake Rs 80.
RULES:  Never promise a delivery time. If the answer isn't in FACTS, say "I'll check with the owner."
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    temperature=0.2,                              # facts and prices: keep it focused
    messages=[{"role": "system", "content": SYSTEM},
              {"role": "user",   "content": "Can I get an eggless cake today evening?"}],
)
print(response.choices[0].message.content)
print(response.usage)

We’d love to make an eggless cake for you, but we need at least 24 hours’ notice, so today evening isn’t possible. Please place your order for tomorrow and let us know the flavor you’d like!
CompletionUsage(completion_tokens=185, prompt_tokens=183, total_tokens=368, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=129, rejected_prediction_tokens=None), prompt_tokens_details=None, queue_time=0.167675505, prompt_time=0.007453152, completion_time=0.38936381, total_time=0.396816962)


In [11]:
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}   # key → value

In [14]:
MENU["donut"]                  # 500
MENU.get("donut", "donuts not available")   # 'not on menu'  (no crash)
#MENU["donut"]

'donuts not available'

In [15]:
order = {"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": True}


In [16]:
order["qty"] * MENU[order["item"]] # 1000


1000

In [17]:
import json
text = json.dumps(order)           # dict → JSON string, to send or save
back = json.loads(text)

In [18]:
text

'{"name": "Ravi", "item": "plum cake", "qty": 2, "eggless": true}'

In [19]:
back

{'name': 'Ravi', 'item': 'plum cake', 'qty': 2, 'eggless': True}

In [20]:
def get_price(item: str, qty: int = 1) -> int:
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [21]:
def get_price(item, qty=1) :
    """Return the total price in rupees for qty of an item on the Sweet Crumbs menu."""
    return MENU[item.lower().strip()] * qty

get_price("cupcake", 6)    # 480
get_price("Plum Cake")

500

In [22]:
a=90
print(a+b)
print('hi')

NameError: name 'b' is not defined

In [23]:
try:
  a=10
  print(a+b)
  print('hi')
except:
  print('b is not defined')


b is not defined


In [24]:
import json, functools
from openai import RateLimitError, APIConnectionError

MODEL = "openai/gpt-oss-120b"
MENU = {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}

SYSTEM = f"""ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.
FACTS:  Menu with prices in rupees: {json.dumps(MENU)}
        Orders need 24 hours' notice. All cakes have eggless versions.
RULES:  Never promise a delivery time. If it isn't in FACTS, say "I'll check with the owner."
        Remember the customer's name and order during the chat.
FORMAT: Under 50 words, friendly, English or Telugu matching the customer."""

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"   [log] {func.__name__}{args} -> {result!r}")
        return result
    return wrapper

@log_call
def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for qty of an item on the menu."""
    return MENU[item.lower().strip()] * qty

history = [{"role": "system", "content": SYSTEM}]
MAX_TURNS = 8

def chat(user_text: str) -> str:
    history.append({"role": "user", "content": user_text})
    recent = [history[0]] + history[1:][-MAX_TURNS:]
    try:
        response = client.chat.completions.create(model=MODEL, messages=recent, temperature=0.3)
        reply = response.choices[0].message.content
    except RateLimitError:
        reply = "We're a little busy. Please try again in a minute."
    except APIConnectionError:
        reply = "I can't reach the server right now. Please try again shortly."
    history.append({"role": "assistant", "content": reply})
    return reply


while True:
    text = input("You: ")
    if text.lower().strip() in ("bye", "exit", "quit"):
        print("Bot: Thank you for visiting Sweet Crumbs!")
        break
    print("Bot:", chat(text))

You: cupcakes unnaya
Bot: Yes, we have cupcakes! 🎂 Each is ₹80 and we also offer an egg‑less version. Just give us a 24‑hour notice for your order. Let me know how many you’d like!
You: telugu lo matladu
Bot: అవును, మా వద్ద కప్‌కేక్‌లు ఉన్నాయి! ఒక్కటి ₹80, ఎగ్‌లెస్ వేరియంట్ కూడా ఉంది. దయచేసి 24 గంటల ముందుగా ఆర్డర్ ఇవ్వండి. మీరు ఎంత quantity కావాలి? మీ పేరు చెప్పండి, తద్వారా నమోదు చేసుకుంటాం.
You: ivala sayanthram ki kavali
Bot: క్షమించండి, అన్ని ఆర్డర్లకు 24 గంటల ముందుగా నోటిస్ అవసరం. దయచేసి మీరు ఎంత కప్‌కేక్‌లు కావాలో, మీ పేరు చెప్పండి; తర్వాత నమోదు చేసుకుంటాం.
You: ayithe vadhu
Bot: సరే! 24 గంటల ముందుగా నోటిస్ అవసరం, కానీ మీరు ఆర్డర్ ఇవ్వడానికి సిద్ధంగా ఉంటే, దయచేసి కావలసిన కప్‌కేక్‌ల సంఖ్య మరియు మీ పేరు చెప్పండి. మీ ఆర్డర్‌ను నమోదు చేసుకుంటాం.
You: bye
Bot: Thank you for visiting Sweet Crumbs!


In [25]:
history


[{'role': 'system',
  'content': 'ROLE:   You are the friendly assistant for Sweet Crumbs Home Bakery.\nFACTS:  Menu with prices in rupees: {"plum cake": 500, "chocolate cake": 650, "cupcake": 80}\n        Orders need 24 hours\' notice. All cakes have eggless versions.\nRULES:  Never promise a delivery time. If it isn\'t in FACTS, say "I\'ll check with the owner."\n        Remember the customer\'s name and order during the chat.\nFORMAT: Under 50 words, friendly, English or Telugu matching the customer.'},
 {'role': 'user', 'content': 'cupcakes unnaya'},
 {'role': 'assistant',
  'content': 'Yes, we have cupcakes! 🎂 Each is ₹80 and we also offer an egg‑less version. Just give us a 24‑hour notice for your order. Let me know how many you’d like!'},
 {'role': 'user', 'content': 'telugu lo matladu'},
 {'role': 'assistant',
  'content': 'అవును, మా వద్ద కప్\u200cకేక్\u200cలు ఉన్నాయి! ఒక్కటి ₹80, ఎగ్\u200cలెస్ వేరియంట్ కూడా ఉంది. దయచేసి 24 గంటల ముందుగా ఆర్డర్ ఇవ్వండి. మీరు ఎంత quantity కావాలి?